# OrbitBench RL sur Colab

Ce notebook fait la partie Colab de l'étude RL. Le modèle joue des exercices d'entraînement, on garde ses bonnes réponses, puis on le ré-entraîne dessus (un « fine-tune »). Le protocole complet est dans `docs/rl_plan.md`. L'évaluation finale se fait ensuite chez vous, pas ici (partie F).

## Avant de commencer
1. Menu **Exécution → Modifier le type d'exécution** : choisissez le GPU **T4** et cochez **RAM élevée**.
2. Lancez les cellules une par une, de haut en bas, avec Maj+Entrée.
3. Gardez cet onglet ouvert tant qu'une cellule tourne.

## L'ordre
| Partie | Quoi | Quand | Durée estimée |
|---|---|---|---|
| A | Démarrage | À chaque session | 15 à 20 min |
| B | Modèle de base, test complet, phase 0 | Une seule fois | 1 h 30 à 2 h |
| C | Tour 0 | Pour T = 0, puis 1, puis 2 | 4 à 5 h |
| D | Tour 1 | Pour T = 0, puis 1, puis 2 | 7 à 9 h |
| E | Fin de session | Chaque fois que vous arrêtez | 1 min |
| F | Évaluation | À la fin, chez vous | plusieurs jours |

Les durées sont des estimations : rien n'a encore tourné sur Colab. La cellule B6 donnera les vraies.

## Si quelque chose se passe mal
- **La session coupe.** Rien d'important n'est perdu : les résultats sont sur votre Drive. Relancez toute la partie A, puis la cellule qui tournait. Elle reprend où elle en était.
- **Une cellule affiche une erreur.** Arrêtez-vous, copiez le message en entier et envoyez-le à Claude.
- **Vous ne savez plus où vous en êtes.** Lancez la cellule A8.

---
# A. Démarrage (à chaque session)
Une session Colab repart de zéro : ces cellules réinstallent tout. Lancez-les toutes, dans l'ordre.

In [ ]:
# A1. Vérifier que le GPU est là. Vous devez voir « Tesla T4 ».
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# A2. Brancher Google Drive. Une fenêtre demande l'autorisation : acceptez.
import os
from google.colab import drive
drive.mount("/content/drive")
RL_DIR = "/content/drive/MyDrive/orbitbench-rl"
os.makedirs(RL_DIR, exist_ok=True)
os.environ["RL_DIR"] = RL_DIR
os.environ["VLLM_BASE_URL"] = "http://localhost:8000/v1"
os.environ["VLLM_API_KEY"] = "local"
print("Dossier de l'étude :", RL_DIR)
print("Il contient :", sorted(os.listdir(RL_DIR)))

In [ ]:
# A3. Récupérer le code depuis GitHub (ou le mettre à jour s'il est déjà là).
import os
REPO = "https://github.com/Guy2Es/orbitbench.git"
if os.path.isdir("/content/orbitbench/.git"):
    !git -C /content/orbitbench pull --ff-only
else:
    !git clone {REPO} /content/orbitbench
%cd /content/orbitbench
assert os.path.exists("scripts/rl_colab.py"), "Le code n'est pas là. Copiez le message ci-dessus et envoyez-le à Claude."
!git log -1 --format="Version du code : %h du %cd" --date=short

In [ ]:
# A4. Installer le banc d'essai (environ 2 minutes). Les avertissements jaunes de pip sont sans importance.
!pip install -q -e ".[harness]" "openai>=3.4.0"
!python -c "import orbitbench, inspect_ai; print('Banc d essai prêt, inspect-ai', inspect_ai.__version__)"

In [ ]:
# A5. Installer vLLM, le serveur qui fait tourner le modèle (5 à 10 minutes).
!python scripts/rl_colab.py install-vllm

In [ ]:
# A6. Installer Unsloth, l'outil de fine-tune (2 à 3 minutes). Lignes reprises du notebook Unsloth pour Qwen3-4B.
!pip install -q sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install -q --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!unsloth install-kernels
!pip install -q --no-deps --upgrade "torchao>=0.16.0"
!pip install -q transformers==4.57.6
!pip install -q --no-deps trl==0.22.2

In [ ]:
# A7. Réglages de l'étude. Relancez cette cellule chaque fois que vous changez une valeur.
BASE = "qwen3-4b"   # nom du modèle de base sur Drive (le « bras 0 »)
BLOCK_N = 144       # taille d'un bloc : 144, ou 96 si la phase 0 le demande (cellule B6)
T = 0               # graine d'entraînement en cours : 0, puis 1, puis 2

def ok(message="La dernière commande a échoué."):
    """Arrête la cellule si la dernière commande lancée avec ! a échoué."""
    code = globals().get("_exit_code", 0)
    assert code == 0, f"{message} (code {code}) Copiez les messages ci-dessus et envoyez-les à Claude."

print(f"Modèle {BASE}, blocs de {BLOCK_N} instances, graine d'entraînement T = {T}")

In [ ]:
# A8. Où en suis-je ? À lancer quand vous voulez : la cellule lit ce qui existe sur Drive.
import os

def etat_bloc(dossier):
    if not os.path.isdir(dossier):
        return "à faire"
    fichier = os.path.join(dossier, "episodes.jsonl")
    n = sum(1 for _ in open(fichier, encoding="utf-8")) if os.path.exists(fichier) else 0
    return "fait" if n >= BLOCK_N * 4 else f"commencé ({n} épisodes sur {BLOCK_N * 4}) : relancer la cellule"

def etat(chemin):
    return "fait" if os.path.exists(chemin) else "à faire"

print("B1 modèle de base sur Drive :", etat(f"{RL_DIR}/models/{BASE}/manifest.json"))
print("B6 phase 0                  :", etat(f"{RL_DIR}/phase0-{BASE}/episodes.jsonl"))
for t in (0, 1, 2):
    print(f"\nGraine T = {t}")
    print("  C1 bloc du tour 0           :", etat_bloc(f"{RL_DIR}/t{t}-r0-d0-base"))
    print("  C2 sélection du tour 0      :", etat(f"{RL_DIR}/sft/t{t}-r0-reward.jsonl"))
    print("  C3 fine-tune reward, tour 0 :", etat(f"{RL_DIR}/models/t{t}r0-reward/manifest.json"))
    print("  C4 fine-tune format, tour 0 :", etat(f"{RL_DIR}/models/t{t}r0-format/manifest.json"))
    print("  D1 bloc du tour 1, reward   :", etat_bloc(f"{RL_DIR}/t{t}-r1-d0-reward"))
    print("  D2 bloc du tour 1, format   :", etat_bloc(f"{RL_DIR}/t{t}-r1-d0-format"))
    print("  D3 sélection et ajout       :", etat(f"{RL_DIR}/sft/t{t}-r01-format.jsonl"))
    print("  D4 fine-tune final reward   :", etat(f"{RL_DIR}/models/t{t}-reward/manifest.json"))
    print("  D5 fine-tune final format   :", etat(f"{RL_DIR}/models/t{t}-format/manifest.json"))

---
# B. Une seule fois : modèle de base, test complet, phase 0
Le test (B2 à B5) vérifie toute la chaîne sur un petit exemple avant de dépenser des heures de GPU. Son modèle est jeté et ne compte dans aucun résultat.

In [ ]:
# B1. Télécharger le modèle de base et le ranger sur Drive (15 à 20 minutes, environ 8 Go).
import os, shutil
if os.path.exists(f"{RL_DIR}/models/{BASE}/manifest.json"):
    print("Le modèle de base est déjà sur Drive : rien à faire.")
else:
    !python scripts/rl_finetune.py export-base --out /content/export/{BASE}
    ok("Le téléchargement a échoué.")
    shutil.copytree(f"/content/export/{BASE}/merged", f"{RL_DIR}/models/{BASE}/merged",
                    dirs_exist_ok=True, ignore=shutil.ignore_patterns(".cache"))
    shutil.copy(f"/content/export/{BASE}/manifest.json", f"{RL_DIR}/models/{BASE}/manifest.json")
    print("Modèle de base rangé sur Drive.")

In [ ]:
# B2. Test 1/4 : démarrer le serveur et jouer 16 épisodes (10 à 20 minutes).
# À voir à la fin : un tableau, avec des appels d'outil (tool_calls) et, si possible, quelques épisodes résolus.
!python scripts/rl_colab.py serve --model {BASE}
ok("Le serveur n'a pas démarré. Lancez la cellule B2 bis et envoyez ce qu'elle affiche.")
!RL_DIR=/content/smoke python scripts/rl_round.py play --arm smoke --tag {BASE} --train-seed 0 --round 0 --first-seed 1950000 --block-n 4 --server vllm --sandbox local

In [ ]:
# B2 bis. Seulement si le serveur pose problème : afficher la fin de son journal.
!tail -n 60 /content/vllm.log

In [ ]:
# B3. Test 2/4 : le texte vu à l'entraînement est-il le même que celui vu par le serveur ? (1 minute)
# À voir à la fin : « OK: the render matches the log ».
!python scripts/rl_colab.py stop
!RL_DIR=/content/smoke python scripts/rl_round.py select --train-seed 0 --round 0 --base-arm smoke --only format --count 16
ok("La sélection du test a échoué.")
!python scripts/rl_finetune.py check --model /content/models/{BASE} --data /content/smoke/sft/t0-r0-format.jsonl
ok("Le contrôle des tokens ne passe pas : il ne faut pas entraîner.")

In [ ]:
# B4. Test 3/4 : un mini fine-tune sur les réponses du test (15 à 30 minutes).
# À voir : un extrait de texte qui ne contient que des réponses du modèle, puis des lignes de « loss », puis des empreintes.
# Si vous voyez « CUDA out of memory », arrêtez-vous et envoyez le message.
!python scripts/rl_colab.py stop
!python scripts/rl_finetune.py train --model /content/models/{BASE} --data /content/smoke/sft/t0-r0-format.jsonl --out /content/smoke/models/smoke-ft --seed 0
ok("Le mini fine-tune a échoué.")

In [ ]:
# B5. Test 4/4 : servir le modèle fine-tuné, jouer 4 épisodes, puis faire le ménage (10 à 15 minutes).
!RL_DIR=/content/smoke python scripts/rl_colab.py serve --model smoke-ft
ok("Le modèle fine-tuné ne se sert pas. Lancez B2 bis et envoyez ce qu'elle affiche.")
!RL_DIR=/content/smoke python scripts/rl_round.py play --arm smoke2 --tag smoke-ft --train-seed 0 --round 0 --first-seed 1950000 --block-n 1 --server vllm --sandbox local
!python scripts/rl_colab.py stop
!rm -rf /content/smoke/models /content/models/smoke-ft /content/export
print("Test complet terminé. Si les quatre cellules B2 à B5 sont passées sans erreur, continuez avec B6.")

In [ ]:
# B6. Phase 0 : mesurer le temps et le coût d'un bloc (20 à 40 minutes, 72 épisodes).
# Avant de lancer : menu Ressources de Colab (en haut à droite), lisez « taux d'utilisation : environ X par heure »
# et écrivez X ci-dessous, avec un point et non une virgule (exemple : 1.44).
UNITS_PER_HOUR = 0.0
assert UNITS_PER_HOUR > 0, "Remplissez UNITS_PER_HOUR avec le taux affiché par Colab, puis relancez la cellule."
!python scripts/rl_colab.py serve --model {BASE}
ok("Le serveur n'a pas démarré.")
!python scripts/rl_round.py phase0 --name {BASE} --tag {BASE} --server vllm --sandbox local
!python scripts/rl_colab.py stop
!python scripts/rl_round.py bakeoff --finetune-ok {BASE} --default {BASE} --parallel 8 --units-per-hour {UNITS_PER_HOUR}

**Arrêtez-vous ici.** Copiez le tableau affiché par B6 et envoyez-le à Claude.

Les deux dernières lignes donnent le temps d'un bloc et le nombre d'unités Colab pour les 9 blocs. Elles disent aussi si les blocs font 144 ou 96 instances. Cette décision doit être écrite dans `docs/rl_plan.md` et poussée sur GitHub **avant** de commencer la partie C. Si la réponse est 96, changez `BLOCK_N` dans la cellule A7 et relancez-la.

La ligne « choice by the rule » ne sert plus : le modèle est déjà fixé.

---
# C. Tour 0 (pour T = 0, puis T = 1, puis T = 2)
Le modèle de base joue un bloc d'exercices, 4 essais par exercice. On en tire deux jeux de réponses de même taille : les réponses justes (bras « reward ») et des réponses bien formées, justes ou fausses (bras « format »). Puis un fine-tune par bras.

Vérifiez d'abord que la cellule A7 affiche la bonne valeur de `T`.

In [ ]:
# C1. Le modèle de base joue le bloc du tour 0 (plusieurs heures ; relancer la cellule si la session coupe).
!python scripts/rl_colab.py serve --model {BASE}
ok("Le serveur n'a pas démarré.")
!python scripts/rl_round.py play --arm base --tag {BASE} --train-seed {T} --round 0 --block-n {BLOCK_N} --server vllm --sandbox local
ok("Le bloc n'est pas fini : relancez cette cellule, elle reprend où elle en était.")
!python scripts/rl_colab.py stop

In [ ]:
# C2. Choisir les réponses gardées pour chaque bras (1 à 2 minutes).
# À voir : deux lignes, « reward » et « format », avec le même nombre d'exemples.
!python scripts/rl_round.py select --train-seed {T} --round 0 --base-arm base
assert globals().get("_exit_code", 0) in (0, 3), "La sélection a échoué. Copiez le message et envoyez-le à Claude."
if globals().get("_exit_code", 0) == 3:
    print("\nMoins de 48 réponses justes : lancez la cellule C2 bis avant de continuer.")

In [ ]:
# C2 bis. SEULEMENT si C2 a dit « only ... rewarded replies (< 48) » : un second bloc, puis une nouvelle sélection.
!python scripts/rl_colab.py serve --model {BASE}
ok("Le serveur n'a pas démarré.")
!python scripts/rl_round.py play --arm base --tag {BASE} --train-seed {T} --round 0 --redraw 1 --block-n {BLOCK_N} --server vllm --sandbox local
ok("Le bloc n'est pas fini : relancez cette cellule.")
!python scripts/rl_colab.py stop
!python scripts/rl_round.py select --train-seed {T} --round 0 --base-arm base
print("\nMême s'il reste moins de 48 réponses justes, on continue avec ce qu'il y a (règle du protocole) : notez le nombre.")

In [ ]:
# C3. Fine-tune du bras « reward », tour 0 (20 à 40 minutes). Ne fait rien s'il est déjà fait.
!python scripts/rl_colab.py stop
!python scripts/rl_colab.py local --model {BASE}
ok("Le modèle de base n'a pas pu être copié depuis Drive.")
!python scripts/rl_finetune.py check --model /content/models/{BASE} --data {RL_DIR}/sft/t{T}-r0-reward.jsonl
ok("Le contrôle des tokens ne passe pas : il ne faut pas entraîner.")
!python scripts/rl_finetune.py train --model /content/models/{BASE} --data {RL_DIR}/sft/t{T}-r0-reward.jsonl --out {RL_DIR}/models/t{T}r0-reward --seed {T}
ok("Le fine-tune a échoué.")

In [ ]:
# C4. Fine-tune du bras « format », tour 0 (20 à 40 minutes). Ne fait rien s'il est déjà fait.
!python scripts/rl_colab.py stop
!python scripts/rl_colab.py local --model {BASE}
ok("Le modèle de base n'a pas pu être copié depuis Drive.")
!python scripts/rl_finetune.py train --model /content/models/{BASE} --data {RL_DIR}/sft/t{T}-r0-format.jsonl --out {RL_DIR}/models/t{T}r0-format --seed {T}
ok("Le fine-tune a échoué.")

---
# D. Tour 1 (même valeur de T que la partie C)
Chacun des deux modèles du tour 0 joue un nouveau bloc. On refait la sélection, on ajoute les réponses des deux tours, et on refait un fine-tune **en repartant du modèle de base**. Ces deux derniers modèles sont ceux qui seront évalués.

In [ ]:
# D1. Le modèle « reward » du tour 0 joue le bloc du tour 1 (plusieurs heures ; relancer si la session coupe).
!python scripts/rl_colab.py serve --model t{T}r0-reward
ok("Le serveur n'a pas démarré.")
!python scripts/rl_round.py play --arm reward --tag t{T}r0-reward --train-seed {T} --round 1 --block-n {BLOCK_N} --server vllm --sandbox local
ok("Le bloc n'est pas fini : relancez cette cellule, elle reprend où elle en était.")
!python scripts/rl_colab.py stop

In [ ]:
# D2. Le modèle « format » du tour 0 joue le bloc du tour 1 (plusieurs heures ; relancer si la session coupe).
!python scripts/rl_colab.py serve --model t{T}r0-format
ok("Le serveur n'a pas démarré.")
!python scripts/rl_round.py play --arm format --tag t{T}r0-format --train-seed {T} --round 1 --block-n {BLOCK_N} --server vllm --sandbox local
ok("Le bloc n'est pas fini : relancez cette cellule, elle reprend où elle en était.")
!python scripts/rl_colab.py stop

In [ ]:
# D3. Sélection du tour 1 à taille égale, puis ajout des tours 0 et 1 (2 à 3 minutes).
import json
!python scripts/rl_round.py select --train-seed {T} --round 1 --base-arm reward --only reward
ok("La sélection du bras reward a échoué.")
N1 = json.load(open(f"{RL_DIR}/sft/t{T}-r1-reward.jsonl.manifest.json"))["examples"]
!python scripts/rl_round.py select --train-seed {T} --round 1 --base-arm format --only format --count {N1}
ok("La sélection du bras format a échoué.")
!python scripts/rl_round.py merge --train-seed {T} --arm reward
ok("L'ajout des deux tours a échoué (reward).")
!python scripts/rl_round.py merge --train-seed {T} --arm format
ok("L'ajout des deux tours a échoué (format).")
# Si une ligne « warning: only ... parsing replies » est apparue, notez-la : les deux bras n'ont pas la même taille.

In [ ]:
# D4. Fine-tune final du bras « reward », depuis le modèle de base, sur les tours 0 et 1 (30 à 60 minutes).
!python scripts/rl_colab.py stop
!python scripts/rl_colab.py local --model {BASE}
ok("Le modèle de base n'a pas pu être copié depuis Drive.")
!python scripts/rl_finetune.py train --model /content/models/{BASE} --data {RL_DIR}/sft/t{T}-r01-reward.jsonl --out {RL_DIR}/models/t{T}-reward --seed {T}
ok("Le fine-tune a échoué.")

In [ ]:
# D5. Fine-tune final du bras « format », depuis le modèle de base, sur les tours 0 et 1 (30 à 60 minutes).
!python scripts/rl_colab.py stop
!python scripts/rl_colab.py local --model {BASE}
ok("Le modèle de base n'a pas pu être copié depuis Drive.")
!python scripts/rl_finetune.py train --model /content/models/{BASE} --data {RL_DIR}/sft/t{T}-r01-format.jsonl --out {RL_DIR}/models/t{T}-format --seed {T}
ok("Le fine-tune a échoué.")

**La graine T est terminée.** Lancez A8 pour le vérifier. Puis, dans la cellule A7, passez `T` à la valeur suivante (0, puis 1, puis 2), relancez A7 et refaites les parties C et D.

Quand les trois graines sont faites, votre Drive contient 7 modèles : `qwen3-4b`, et `t0-reward`, `t0-format`, `t1-reward`, `t1-format`, `t2-reward`, `t2-format`.

---
# E. Fin de session
À lancer chaque fois que vous arrêtez, pour ne plus consommer d'unités. La cellule attend que Drive ait tout enregistré, puis éteint la machine.

In [ ]:
# E1. Arrêter le serveur, finir l'écriture sur Drive, éteindre la machine Colab.
!python scripts/rl_colab.py stop
from google.colab import drive, runtime
drive.flush_and_unmount()
print("Drive est à jour. La machine s'éteint.")
runtime.unassign()

---
# F. À la maison : l'évaluation (pas dans Colab)
L'évaluation a besoin de Docker et de GMAT, que Colab n'a pas. Elle se fait sur votre PC, avec la plage de test privée qui est dans votre fichier `.env`. Demandez à Claude de vous guider à ce moment-là. En résumé :

1. Télécharger depuis Drive le dossier `merged` de chacun des 7 modèles.
2. Les convertir pour Ollama, tous de la même façon : `python scripts/rl_gguf.py --model-dir <dossier merged> --name <nom> --llama-cpp <dossier llama.cpp> --template-from qwen3-4b-instruct-16k`
3. Vérifier qu'Ollama présente le texte au modèle comme à l'entraînement (`scripts/rl_finetune.py check` sur quelques épisodes joués chez vous).
4. Évaluer chaque modèle : `python scripts/rl_round.py evaluate --arm <nom> --tag <nom>` (environ 11 h par modèle sur la GTX 1070).
5. Comparer : `python -m orbitbench.harness compare --base <dossier du modèle de base> --arm <dossiers des trois modèles reward>`